# Foundation Minifigs — Silver Layer

Reads current minifig records from `lego_brickbase.bronze.raw_minifigs` and loads them into the `lego_brickbase.silver.foundation_minifigs` Delta table.

## What this notebook does

1. **Read** — Selects all current, non-deleted records from the bronze `raw_minifigs` table.
2. **Transform** — Renames `set_num` to `minifig_key`, retains business columns (`set_num`, `name`, `num_parts`, `set_img_url`, `set_url`, `last_modified_dt`, `valid_from`), and derives a `valid_from_date` column used for partitioning.
3. **Write** — Overwrites the Delta table at `/Volumes/lego_brickbase/silver/delta_volume/minifigs`, partitioned by `valid_from_date`.
4. **Register** — Creates the catalog table `lego_brickbase.silver.foundation_minifigs` in Unity Catalog if it does not already exist.

## Imports and Configuration

Import Spark functions needed for the transformation. `BRONZE_TABLE` is the source; `DELTA_TABLE_PATH` is the physical Delta location on the Unity Catalog external volume; `CATALOG_TABLE` is the Unity Catalog table reference.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date


BRONZE_TABLE     = "lego_brickbase.bronze.raw_minifigs"
DELTA_TABLE_PATH = "/Volumes/lego_brickbase/silver/delta_volume/minifigs"
CATALOG_TABLE    = "lego_brickbase.silver.foundation_minifigs"

## Read and Transform

Filter the bronze table to current, non-deleted records, then rename `set_num` to `minifig_key` and derive `valid_from_date` (the date portion of `valid_from`) for use as the Delta partition column.

In [ ]:
df_source = (
    spark.table(BRONZE_TABLE)
    .filter((col("is_current") == True) & (col("is_deleted") == False))
)

df_foundation = (
    df_source
    .select(
        col("set_num").alias("minifig_key"),
        col("set_num").alias("minifig_set_number"),
        col("name").alias("minifigs_name"),
        col("num_parts").alias("number_of_parts"),
        col("set_img_url").alias("minifig_set_image_url"),
        col("set_url").alias("minifig_set_url"),
        col("last_modified_dt").alias("source_last_modified_date"),
        col("valid_from"),
    )
    .withColumn("valid_from_date", to_date(col("valid_from")))
)

df_foundation.printSchema()
df_foundation.display(10, truncate=False)

## Write to Delta Volume and Register Catalog Table

Overwrite the Delta table at the silver volume path, partitioned by `valid_from_date`. Then register it in Unity Catalog as `lego_brickbase.silver.foundation_minifigs` if it does not already exist.

In [ ]:
(
    df_foundation
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("valid_from_date")
    .save(DELTA_TABLE_PATH)
)

row_count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
print(f"Rows written to Delta table: {row_count:,}")

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG_TABLE}
    AS SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")
print(f"Catalog table ready: {CATALOG_TABLE}")